In [1]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import gensim.downloader as api
import random

In [2]:
df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [3]:
# Embedding
from gensim.models import Word2Vec

model_CBOW = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

model_S = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

In [4]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_CBOW.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['unwittingly', 'anticipation', 'diagnostic', 'delicious', 'Police', 'witch-craft', 'butcher', 'biologic', 'polders', 'thanking', 'aspirations', 'racism', 'downtown', 'sarcasm', 'roach', 'judgements', '70th', 'fishy', 'H.S.S', 'contempt', 'Mozart', 'Freud', 'recommend', 'Essay', 'flimsy', 'snob', 'petty-minded', 'Jaya', 'parents/friends', 'expired']


In [5]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_S.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['unwittingly', 'anticipation', 'diagnostic', 'delicious', 'Police', 'witch-craft', 'butcher', 'biologic', 'polders', 'thanking', 'aspirations', 'racism', 'downtown', 'sarcasm', 'roach', 'judgements', '70th', 'fishy', 'H.S.S', 'contempt', 'Mozart', 'Freud', 'recommend', 'Essay', 'flimsy', 'snob', 'petty-minded', 'Jaya', 'parents/friends', 'expired']


In [6]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_CBOW.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage CBOW: {token_coverage:.2f}%")

Token-level coverage CBOW: 97.18%


In [7]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_S.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage S: {token_coverage:.2f}%")

Token-level coverage S: 97.18%


In [8]:
print(model_CBOW.wv.most_similar("leukemia", topn=10))
print(model_S.wv.most_similar("leukemia", topn=10))

[('suffered', 0.9494734406471252), ('stealing', 0.9394954442977905), ('disease', 0.9372707605361938), ('brother-in-law', 0.933728039264679), ('wristwatch', 0.9330476522445679), ('sick', 0.9296715259552002), ('former', 0.9292072653770447), ('empty', 0.9284663796424866), ('seriously', 0.9244797229766846), ('dad', 0.923152506351471)]
[('gradmother', 0.9731012582778931), ('lung', 0.9580507874488831), ('bewitched', 0.9521536231040955), ('thirteen', 0.9482303261756897), ('paternal', 0.9457148313522339), ('cerebral', 0.9443697929382324), ('eleven', 0.9424349665641785), ('tragically', 0.9418221712112427), ('collapsed', 0.9413235783576965), ('warning', 0.9343069791793823)]


In [9]:
print([word for word in model_vocab if len(word) <= 2])
print([word for word in dataset_vocab if len(word) <= 2])

['ca', 'if', 'us', 'B+', '``', 'go', '.', 'Ye', '/', 'up', '&', '18', '22', 'km', 'An', 'do', '12', 'My', 'At', 'no', ':', 'at', 'be', 'on', 'He', ',', 'Mr', 'as', 'PY', '14', 'If', 'II', '-', '20', 'Oh', 'is', 'TE', '1', 'Ca', 'aa', '84', 'D', 'Km', 'F4', 'my', 'LP', 'B', "'s", 'e', "''", 'A', '$', 'I.', 'ex', 'OK', '50', '6', 'No', 'ok', 'm', 'As', 'TA', '3', 'we', "'", '16', 'On', '(', 'X', 'in', '19', "'d", '[', '4', '!', ';', '2A', '13', '7', '15', '11', '9', 'To', 'an', 'C+', "'m", '21', 'In', 'of', '%', '60', '?', 'TV', 'so', 'he', 'to', '30', 'So', 'I', ')', '35', 'C', 'We', 'By', 'Is', 'i', 'by', 'am', ']', 'NO', 'me', '40', '10', 'or', '8', '80', '17', '5', 'it', 'a', '2', 'wo', 'Of', 'pm', 'It']
['ca', 'if', 'us', 'B+', '``', 'NH', 'go', 't', '.', 'CV', 'NB', 'Ye', '--', '/', 'up', '&', '18', '22', 'UC', 'km', 'An', 'do', '12', 'My', 'At', 'El', 'la', 'no', ':', 'at', '24', 'be', 'on', 'He', ',', 'Mr', 'as', 'PY', 'Or', '14', 'If', 'II', 'WD', 'ne', 'E', 's', 'Rs', 'P3', 'aI

In [10]:
# Pooling using mean
def mean_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.mean(vectors, axis=0)

# Pooling using max
def max_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_mean_cbow = np.vstack([
    mean_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_mean_skipgram = np.vstack([
    mean_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

X_max_cbow = np.vstack([
    max_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_max_skipgram = np.vstack([
    max_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

# Vector shapes should be the same
print(X_mean_cbow.shape)
print(X_mean_skipgram.shape)
print(X_max_cbow.shape)
print(X_max_skipgram.shape)


(7516, 100)
(7516, 100)
(7516, 100)
(7516, 100)


In [11]:
# Training classifiers
y = df["emotion"]

implementations = {
    "CBOW + mean": X_mean_cbow,
    "Skip-gram + mean": X_mean_skipgram,
    "CBOW + max": X_max_cbow,
    "Skip-gram + max": X_max_skipgram,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    results[name] = scores
    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

CBOW + mean: mean=0.3493, std=0.0127
Skip-gram + mean: mean=0.4266, std=0.0106
CBOW + max: mean=0.3260, std=0.0072
Skip-gram + max: mean=0.3733, std=0.0083


In [12]:
# Now trying the same but using Glove instead
glove = api.load("glove-wiki-gigaword-100")
vector_size = glove.vector_size

# Glove works in lowercase
df["glove_tokens"] = [
    [token.lower() for token in tokens]
    for tokens in df["tokens"]
]

# Pooling functions but with glove:
def glove_mean_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.mean(vectors, axis=0)

def glove_max_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_glove_mean = np.vstack([
    glove_mean_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

X_glove_max = np.vstack([
    glove_max_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

print(X_glove_mean.shape)
print(X_glove_max.shape)

[==================================================] 100.0% 128.1/128.1MB downloaded
(7516, 100)
(7516, 100)


In [13]:
# Glove coverage
glove_covered_tokens = sum(
    token in glove
    for tokens in df["glove_tokens"]
    for token in tokens
)

glove_total_tokens = sum(
    len(tokens)
    for tokens in df["glove_tokens"]
)

print(
    f"GloVe token coverage: "
    f"{glove_covered_tokens / glove_total_tokens:.2%}"
)

GloVe token coverage: 99.37%


In [14]:
# Training classifiers
y = df["emotion"]

implementations = {
    "GloVe mean": X_glove_mean,
    "GloVe max": X_glove_max,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

GloVe mean: mean=0.4864, std=0.0164
GloVe max: mean=0.4211, std=0.0099


In [23]:
import sys
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [33]:
# Willen jullie dit testen a.u.b.?
sys.path.insert(0, os.path.abspath(".."))
import jax
import numpyro
import part2
import itertools

glove = api.load("glove-wiki-gigaword-100", return_path=True)
trained_emb_model = part2.clslib.PretrainedEmbeddings(glove, 100, skip_first_line=True)
tokenizer_for_lstm = part2.clslib.BasicTokenizer(
    True, False, False, 'english', trained_emb_model)

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

pairs=list(zip(df["content"], df["emotion"].map(list(emotions).index)))
splits = cross_validation_split(pairs, k=5)

part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.928 seconds
Dataloader done! Time taken: 0.035 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9323 | train acc: 0.1737 | val loss: 1.8451 | val acc: 0.2513
Epoch 02 | train loss: 1.8007 | train acc: 0.2791 | val loss: 1.7346 | val acc: 0.3344
Epoch 03 | train loss: 1.6984 | train acc: 0.3283 | val loss: 1.6449 | val acc: 0.3484
Epoch 04 | train loss: 1.6071 | train acc: 0.3698 | val loss: 1.6236 | val acc: 0.3584
Epoch 05 | train loss: 1.5337 | train acc: 0.4029 | val loss: 1.4887 | val acc: 0.4215
Epoch 06 | train loss: 1.4480 | train acc: 0.4431 | val loss: 1.4343 | val acc: 0.4408
Epoch 07 | train loss: 1.4075 | train acc: 0.4573 | val loss: 1.4084 | val acc: 0.4694
Epoch 08 | train loss: 1.3619 | train acc: 0.4799 | val loss: 1.3735 | val acc: 0.4787
Epoch 09 | train loss: 1.3378 | train acc: 0.4787 | val loss: 1.3499 | val acc: 0.4914
Epoch 10 | train loss: 1.3154 | train acc: 0.5043 | val loss: 1.3465 | val acc: 0.4914
Epoch 11 | train loss: 1.2791 | train acc: 0.5096 | val loss: 1.3594 | val acc: 0.4960
Epoch 12 | train loss: 1.2468 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9384 | train acc: 0.1628 | val loss: 1.9042 | val acc: 0.2236
Epoch 02 | train loss: 1.8130 | train acc: 0.2671 | val loss: 1.7589 | val acc: 0.3114
Epoch 03 | train loss: 1.6616 | train acc: 0.3521 | val loss: 1.5904 | val acc: 0.4079
Epoch 04 | train loss: 1.5364 | train acc: 0.4041 | val loss: 1.5145 | val acc: 0.4271
Epoch 05 | train loss: 1.4516 | train acc: 0.4510 | val loss: 1.4715 | val acc: 0.4444
Epoch 06 | train loss: 1.3953 | train acc: 0.4805 | val loss: 1.4156 | val acc: 0.4770
Epoch 07 | train loss: 1.3606 | train acc: 0.4951 | val loss: 1.4114 | val acc: 0.4731
Epoch 08 | train loss: 1.3131 | train acc: 0.5147 | val loss: 1.3747 | val acc: 0.5110
Epoch 09 | train loss: 1.2719 | train acc: 0.5287 | val loss: 1.3672 | val acc: 0.5090
Epoch 10 | train loss: 1.2468 | train acc: 0.5420 | val loss: 1.3300 | val acc: 0.5263
Epoch 11 | train loss: 1.2159 | train acc: 0.5595 | val loss: 1.3293 | val acc: 0.5176
Epoch 12 | train loss: 1.1856 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9222 | train acc: 0.1911 | val loss: 1.8489 | val acc: 0.2502
Epoch 02 | train loss: 1.7451 | train acc: 0.3127 | val loss: 1.6963 | val acc: 0.3260
Epoch 03 | train loss: 1.6352 | train acc: 0.3630 | val loss: 1.6392 | val acc: 0.3673
Epoch 04 | train loss: 1.5559 | train acc: 0.3975 | val loss: 1.5741 | val acc: 0.3859
Epoch 05 | train loss: 1.4983 | train acc: 0.4141 | val loss: 1.5338 | val acc: 0.4125
Epoch 06 | train loss: 1.4491 | train acc: 0.4404 | val loss: 1.4928 | val acc: 0.4471
Epoch 07 | train loss: 1.4052 | train acc: 0.4648 | val loss: 1.4222 | val acc: 0.4737
Epoch 08 | train loss: 1.3589 | train acc: 0.4823 | val loss: 1.3655 | val acc: 0.4977
Epoch 09 | train loss: 1.3228 | train acc: 0.4981 | val loss: 1.3478 | val acc: 0.4943
Epoch 10 | train loss: 1.2899 | train acc: 0.5082 | val loss: 1.2985 | val acc: 0.5203
Epoch 11 | train loss: 1.2577 | train acc: 0.5270 | val loss: 1.2781 | val acc: 0.5356
Epoch 12 | train loss: 1.2324 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9329 | train acc: 0.1778 | val loss: 1.8698 | val acc: 0.2322
Epoch 02 | train loss: 1.7986 | train acc: 0.2736 | val loss: 1.7563 | val acc: 0.3094
Epoch 03 | train loss: 1.7012 | train acc: 0.3240 | val loss: 1.6499 | val acc: 0.3573
Epoch 04 | train loss: 1.6161 | train acc: 0.3730 | val loss: 1.5902 | val acc: 0.3699
Epoch 05 | train loss: 1.5448 | train acc: 0.4046 | val loss: 1.5295 | val acc: 0.4079
Epoch 06 | train loss: 1.4786 | train acc: 0.4442 | val loss: 1.4544 | val acc: 0.4258
Epoch 07 | train loss: 1.4181 | train acc: 0.4695 | val loss: 1.4290 | val acc: 0.4518
Epoch 08 | train loss: 1.3576 | train acc: 0.4899 | val loss: 1.3866 | val acc: 0.4677
Epoch 09 | train loss: 1.3232 | train acc: 0.5094 | val loss: 1.3752 | val acc: 0.4704
Epoch 10 | train loss: 1.2815 | train acc: 0.5267 | val loss: 1.3288 | val acc: 0.5037
Epoch 11 | train loss: 1.2570 | train acc: 0.5400 | val loss: 1.3131 | val acc: 0.5096
Epoch 12 | train loss: 1.2227 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 6, 3, 0], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([5, 0, 0, ..., 2, 0, 2], shape=(1503,))),
              (3, array([0, 4, 6, ..., 3, 1, 4], shape=(1503,))),
              (4, array([5, 2, 6, ..., 0, 0, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.5285904255319149),
              (1, 0.5489021956087824),
              (2, 0.541583499667332),
              (3, 0.5708582834331337),
              (4, 0.5602129075182968)])}